# 04 — External API Clients (VirusTotal · Shodan · NVD/CVE)

**Owner:** Marston Ward (AIE / Team Lead) · **Course:** AAI-510 · Group 8

This notebook builds and demonstrates the three external **threat-intelligence
client wrappers** the SOC triage agent calls during enrichment. Each wrapper is
independent and lives in `src/soc_agent/api_clients.py`.

| Tool | Source | Default mode | Returns (normalized) |
|------|--------|--------------|----------------------|
| `check_ip_reputation()` | VirusTotal v3 | **MOCK** | `{indicator, malicious, suspicious, harmless, verdict, ...}` |
| `lookup_exposed_ports()` | Shodan | **MOCK** | `{indicator, ports[], banners[], os, ...}` |
| `get_cve_context()` | NIST **NVD** (keyless) | **REAL when online**, mock fallback | `[{cve_id, cvss, summary}, ...]` |

### Why MOCK_MODE is the default
- **No API keys are available** for VirusTotal or Shodan.
- The gold layer's `host_ip` column actually holds a **Windows hostname**
  (e.g. `WORKSTATION5`), **not a routable IP**, so a live VT/Shodan lookup would
  be meaningless. Mock fixtures keep the demo realistic and reproducible.
- **NVD is keyless**, so we make **real calls when online** and fall back to a
  static CVE fixture when offline.

### Ownership note — `get_cve_context` (NVD)
Per the proposal, `get_cve_context()` is **officially Sai's (DE)** tool, but it
was **never registered as a Unity Catalog function**. We implement it here on the
agent side as a keyless NVD Python tool. **It should be migrated to a UC SQL/Python
function later** so the data layer owns it.

### Engineering
All wrappers handle errors gracefully: **timeouts**, **bounded retries with
exponential backoff** on transient HTTP statuses (429/5xx), and they **never raise
into the agent loop** — on failure they return a dict containing an `error` key.


In [1]:
# --- bootstrap: make src/soc_agent importable + default to MOCK_MODE ---
import os, sys
from pathlib import Path

# Walk up to the repo root (the dir that contains src/soc_agent).
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "soc_agent").exists():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate src/soc_agent from " + str(_here))

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)  # so ./mlruns and relative paths are consistent

# Default to zero-creds mock mode unless the grader set real env vars.
os.environ.setdefault("SOC_MOCK_MODE", "1")
os.environ.setdefault("MLFLOW_TRACKING_URI", "file:./mlruns")

from soc_agent import config
SETTINGS = config.get_settings()
print("Repo root        :", REPO_ROOT)
print("MOCK_MODE        :", SETTINGS.mock_mode)
print("LLM provider     :", SETTINGS.llm_provider, "(effective:", SETTINGS.effective_llm_provider() + ")")
print("Live Databricks  :", SETTINGS.use_live_databricks())


Repo root        : /Users/marston.ward/Documents/GitHub/msaai-510-group8-soc-triage-agent
MOCK_MODE        : True
LLM provider     : databricks (effective: mock)
Live Databricks  : False


In [2]:
from soc_agent import api_clients
from soc_agent.mocks import SAMPLE_EVENTS
import json

clients = api_clients.build_clients(SETTINGS)
print("Clients:", list(clients))
print("VT mock_mode    :", clients["virustotal"].mock_mode)
print("Shodan mock_mode:", clients["shodan"].mock_mode)


Clients: ['virustotal', 'shodan', 'nvd']
VT mock_mode    : True
Shodan mock_mode: True


## VirusTotal — `check_ip_reputation()`

Returns malicious/suspicious vote counts and a simple verdict for an indicator.
In mock mode a couple of hosts are made to look malicious so enrichment is
meaningful downstream.

In [3]:
vt = clients["virustotal"]
for host in ["WORKSTATION5", "WORKSTATION6", "FILESRV1"]:
    print(host, "->", json.dumps(vt.check_ip_reputation(host)))


WORKSTATION5 -> {"indicator": "WORKSTATION5", "malicious": 0, "suspicious": 1, "harmless": 62, "undetected": 12, "reputation": 5, "verdict": "clean", "source": "mock"}
WORKSTATION6 -> {"indicator": "WORKSTATION6", "malicious": 7, "suspicious": 3, "harmless": 40, "undetected": 12, "reputation": -34, "verdict": "malicious", "source": "mock"}
FILESRV1 -> {"indicator": "FILESRV1", "malicious": 7, "suspicious": 3, "harmless": 40, "undetected": 12, "reputation": -34, "verdict": "malicious", "source": "mock"}


## Shodan — `lookup_exposed_ports()`

Returns open ports and service banners. Banners feed the NVD CVE lookup
(the proposal's documented Shodan → CVE dependency).

In [4]:
shodan = clients["shodan"]
for host in ["FILESRV1", "DC01", "WORKSTATION2"]:
    print(host, "->", json.dumps(shodan.lookup_exposed_ports(host)))


FILESRV1 -> {"indicator": "FILESRV1", "ports": [445, 3389, 139], "banners": ["SMB", "RDP", "NetBIOS"], "os": "Windows", "source": "mock"}
DC01 -> {"indicator": "DC01", "ports": [88, 389, 636, 445], "banners": ["Kerberos", "LDAP", "LDAPS", "SMB"], "os": "Windows", "source": "mock"}
WORKSTATION2 -> {"indicator": "WORKSTATION2", "ports": [], "banners": [], "os": "Windows", "source": "mock"}


## NVD / CVE — `get_cve_context()` (keyless, real when online)

Queries the NIST NVD 2.0 REST API for CVEs with CVSS ≥ 7.0 matching a software /
service keyword. If the network is unavailable the wrapper transparently falls
back to a static CVE fixture, so this cell **always returns data with zero creds**.

In [5]:
nvd = clients["nvd"]
for kw in ["RDP", "SMB", "LDAP"]:
    cves = nvd.get_cve_context(kw, limit=3)
    print(f"{kw}: " + ", ".join(f"{c['cve_id']}(cvss {c['cvss']})" for c in cves[:3])
          + f"   [source={cves[0].get('source','nvd')}]")


RDP: CVE-2001-1158(cvss 7.5)   [source=nvd]


SMB: CVE-1999-0495(cvss 10.0)   [source=nvd]
LDAP: CVE-1999-0385(cvss 10.0), CVE-1999-0895(cvss 7.5)   [source=nvd]


## Graceful error handling

Every wrapper returns a dict even on failure — it never raises into the agent.
Below we force VirusTotal into *live* mode with **no key**; the wrapper attempts a
call and returns a normalized dict with an `error`/`verdict=unknown` field rather
than crashing. (This makes a network attempt; skip if fully offline.)

In [6]:
import os
demo_settings = config.Settings.from_env()
demo_settings.mock_mode = False
demo_settings.vt_api_key = None
vt_live = api_clients.VirusTotalClient(demo_settings, mock_mode=False)
try:
    out = vt_live.check_ip_reputation("8.8.8.8")
except Exception as e:  # should not happen — wrapper catches internally
    out = {"unexpected_exception": str(e)}
print("Graceful result:", json.dumps(out))
print("\nKey takeaway: the agent always receives a dict, never an exception.")


Graceful result: {"indicator": "8.8.8.8", "source": "virustotal", "error": "HTTP 401", "verdict": "unknown"}

Key takeaway: the agent always receives a dict, never an exception.


### Summary

- Three independent client wrappers, all returning small normalized dicts.
- VirusTotal + Shodan are **mocked by default** (no keys; hostname-not-IP reality).
- NVD makes **real keyless calls when online**, with a static fallback offline.
- `get_cve_context` is **Sai's tool by ownership** — migrate to UC later.

Next: `03_agent_loop.ipynb` wires these clients (plus Sai's gold functions) into
the LangGraph ReAct agent.